# 🎬 Hybrid Recommender System for OTT Streaming Platform

## Netflix-like Recommendation Engine

This notebook demonstrates a production-ready hybrid recommender system combining:
- **Collaborative Filtering** (SVD Matrix Factorization)
- **Content-Based Filtering** (TF-IDF + Cosine Similarity)
- **Popularity-Based Filtering** (Trending & Most Watched)

---

### 📚 Table of Contents
1. [Introduction & Theory](#introduction)
2. [System Architecture](#architecture)
3. [Data Preparation](#data)
4. [Individual Models](#models)
5. [Hybrid System](#hybrid)
6. [Evaluation](#evaluation)
7. [Real-World Improvements](#improvements)

---

## 1. Introduction & Theory <a id="introduction"></a>

### Why Hybrid Recommender for OTT Platforms?

#### Individual Approach Limitations:

**Collaborative Filtering:**
- ✅ Discovers patterns from user behavior
- ✅ Finds similar users automatically
- ❌ Cold-start problem (new users/items)
- ❌ Sparsity issues (most users rate few items)

**Content-Based Filtering:**
- ✅ Works for new items immediately
- ✅ Explainable recommendations
- ❌ Limited diversity (filter bubble)
- ❌ Requires good metadata

**Popularity-Based:**
- ✅ Simple and effective
- ✅ Works for cold-start users
- ❌ No personalization
- ❌ Rich get richer problem

#### Hybrid Approach Benefits:
1. **Overcomes individual weaknesses**
2. **Better cold-start handling**
3. **Balanced exploration vs exploitation**
4. **Higher accuracy and user satisfaction**

### Cold-Start Problem Solutions:

**New Users:**
- Show popular/trending content
- Ask for genre preferences
- Use content-based on initial selections
- Gradually incorporate collaborative filtering

**New Content:**
- Use metadata for content-based matching
- Initial popularity from early viewers
- Promote to users with matching preferences

---

## 2. System Architecture <a id="architecture"></a>

```
┌─────────────────────────────────────────────────────────────────┐
│                    USER REQUEST (User ID)                        │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│                   HYBRID RECOMMENDATION ENGINE                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ┌──────────────────┐  ┌──────────────────┐  ┌──────────────┐ │
│  │  Collaborative   │  │  Content-Based   │  │  Popularity  │ │
│  │    Filtering     │  │    Filtering     │  │    Based     │ │
│  │                  │  │                  │  │              │ │
│  │  Weight: 0.5     │  │  Weight: 0.3     │  │  Weight: 0.2 │ │
│  └────────┬─────────┘  └────────┬─────────┘  └──────┬───────┘ │
│           │                     │                    │          │
│           └─────────────────────┼────────────────────┘          │
│                                 ▼                                │
│                    ┌────────────────────────┐                   │
│                    │   WEIGHTED COMBINATION │                   │
│                    └────────────┬───────────┘                   │
│                                 ▼                                │
│                    ┌────────────────────────┐                   │
│                    │  TOP-K RECOMMENDATIONS │                   │
│                    └────────────────────────┘                   │
└─────────────────────────────────────────────────────────────────┘
```

### Hybrid Scoring Formula:

$$\text{Final Score} = w_1 \times \text{CF Score} + w_2 \times \text{CB Score} + w_3 \times \text{Pop Score}$$

Where:
- $w_1 = 0.5$ (Collaborative Filtering)
- $w_2 = 0.3$ (Content-Based)
- $w_3 = 0.2$ (Popularity)
- $w_1 + w_2 + w_3 = 1.0$

---

## 3. Data Preparation <a id="data"></a>

In [ ]:
# Install required packages (run this in Google Colab)
!pip install pandas numpy scikit-learn scikit-surprise matplotlib seaborn -q

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")

In [ ]:
# Import custom modules
import sys
sys.path.append('../src')

from data_preprocessing import DataPreprocessor
from collaborative_filtering import CollaborativeFiltering
from content_based_filtering import ContentBasedFiltering
from popularity_based import PopularityBasedFiltering
from hybrid_recommender import HybridRecommender
from evaluation import RecommenderEvaluator

print("✅ Custom modules imported successfully")

### 3.1 Create Sample Dataset

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Create sample data
users_df, movies_df, interactions_df = preprocessor.create_sample_data()

print("\n📊 Dataset Overview:")
print(f"Users: {len(users_df)}")
print(f"Movies: {len(movies_df)}")
print(f"Interactions: {len(interactions_df)}")

In [ ]:
# Display sample data
print("\n👥 Sample Users:")
display(users_df.head())

print("\n🎬 Sample Movies:")
display(movies_df.head())

print("\n⭐ Sample Interactions:")
display(interactions_df.head())

### 3.2 Feature Engineering

In [ ]:
# Engineer features
preprocessor.engineer_features()

# Get updated dataframes
users_df = preprocessor.users_df
movies_df = preprocessor.movies_df
interactions_df = preprocessor.interactions_df

print("\n✅ Feature engineering completed")
print("\nNew features:")
print(f"Users: {list(users_df.columns)}")
print(f"Movies: {list(movies_df.columns)}")
print(f"Interactions: {list(interactions_df.columns)}")

### 3.3 Exploratory Data Analysis

In [ ]:
# Get statistics
stats = preprocessor.get_data_stats()

print("\n📈 Dataset Statistics:")
for key, value in stats.items():
    if key != 'rating_distribution':
        print(f"  {key}: {value}")

In [ ]:
# Visualize rating distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Rating distribution
interactions_df['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Interactions per user
user_interactions = interactions_df.groupby('user_id').size()
axes[1].hist(user_interactions, bins=50, color='coral', edgecolor='black')
axes[1].set_title('Interactions per User', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Interactions')
axes[1].set_ylabel('Number of Users')

# Interactions per movie
movie_interactions = interactions_df.groupby('movie_id').size()
axes[2].hist(movie_interactions, bins=50, color='lightgreen', edgecolor='black')
axes[2].set_title('Interactions per Movie', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Number of Interactions')
axes[2].set_ylabel('Number of Movies')

plt.tight_layout()
plt.show()

### 3.4 Train-Test Split

In [ ]:
# Split data for evaluation
train_df, test_df = preprocessor.train_test_split(test_size=0.2)

print(f"\n✅ Data split completed:")
print(f"  Training set: {len(train_df)} interactions")
print(f"  Test set: {len(test_df)} interactions")

---

## 4. Individual Models <a id="models"></a>

### 4.1 Collaborative Filtering (SVD)

In [ ]:
# Train collaborative filtering model
cf_model = CollaborativeFiltering(n_factors=50)
cf_model.fit(train_df)

# Get recommendations for user 1
cf_recommendations = cf_model.get_user_recommendations(user_id=1, n_recommendations=10)

print("\n🎬 Top 10 CF Recommendations for User 1:")
for i, (movie_id, rating) in enumerate(cf_recommendations, 1):
    movie_title = movies_df[movies_df['movie_id'] == movie_id]['title'].values[0]
    print(f"{i}. {movie_title} (ID: {movie_id}) - Predicted Rating: {rating:.2f}")

### 4.2 Content-Based Filtering (TF-IDF)

In [ ]:
# Train content-based filtering model
cb_model = ContentBasedFiltering(max_features=1000)
cb_model.fit(movies_df)

# Find similar movies to movie 1
similar_movies = cb_model.get_similar_movies(movie_id=1, n_similar=5)

print("\n🎬 Top 5 Movies Similar to Movie 1:")
movie_1_title = movies_df[movies_df['movie_id'] == 1]['title'].values[0]
print(f"Reference: {movie_1_title}\n")

for i, (movie_id, similarity) in enumerate(similar_movies, 1):
    movie_title = movies_df[movies_df['movie_id'] == movie_id]['title'].values[0]
    movie_genre = movies_df[movies_df['movie_id'] == movie_id]['genre'].values[0]
    print(f"{i}. {movie_title} (ID: {movie_id})")
    print(f"   Genre: {movie_genre}")
    print(f"   Similarity: {similarity:.3f}\n")

### 4.3 Popularity-Based Filtering

In [ ]:
# Train popularity-based filtering
pop_model = PopularityBasedFiltering(recency_days=30)
pop_model.fit(movies_df, train_df)

# Get popular movies
popular_movies = pop_model.get_popular_movies(n_recommendations=10)

print("\n🔥 Top 10 Popular Movies:")
for i, (movie_id, score) in enumerate(popular_movies, 1):
    movie_title = movies_df[movies_df['movie_id'] == movie_id]['title'].values[0]
    movie_genre = movies_df[movies_df['movie_id'] == movie_id]['genre'].values[0]
    print(f"{i}. {movie_title}")
    print(f"   Genre: {movie_genre}")
    print(f"   Popularity Score: {score:.3f}\n")

In [ ]:
# Get trending movies
trending_movies = pop_model.get_trending_movies(n_recommendations=10)

print("\n📈 Top 10 Trending Movies:")
for i, (movie_id, score) in enumerate(trending_movies, 1):
    movie_title = movies_df[movies_df['movie_id'] == movie_id]['title'].values[0]
    print(f"{i}. {movie_title} - Trending Score: {score:.3f}")

---

## 5. Hybrid Recommender System <a id="hybrid"></a>

In [ ]:
# Initialize and train hybrid recommender
hybrid_recommender = HybridRecommender(
    cf_weight=0.5,
    cb_weight=0.3,
    pop_weight=0.2,
    n_factors=50,
    max_features=1000
)

hybrid_recommender.fit(users_df, movies_df, train_df)

### 5.1 Get Recommendations for Established User

In [ ]:
# Get recommendations for user with interaction history
user_id = 1
recommendations = hybrid_recommender.recommend(user_id, n_recommendations=10)

print(f"\n{'='*80}")
print(f"🎬 TOP 10 HYBRID RECOMMENDATIONS FOR USER {user_id}")
print(f"{'='*80}\n")

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec['title']}")
    print(f"   Genre: {rec['genre']}")
    print(f"   Year: {rec['release_year']}")
    print(f"   Language: {rec['language']}")
    print(f"   Hybrid Score: {rec['hybrid_score']:.3f}")
    print(f"   Component Scores:")
    print(f"     - Collaborative: {rec['cf_score']:.3f}")
    print(f"     - Content-Based: {rec['cb_score']:.3f}")
    print(f"     - Popularity: {rec['pop_score']:.3f}")
    print(f"   Explanation: {rec['explanation']}")
    print()

### 5.2 Visualize Score Components

In [ ]:
# Visualize score breakdown
top_5_recs = recommendations[:5]

titles = [rec['title'][:20] + '...' if len(rec['title']) > 20 else rec['title'] 
          for rec in top_5_recs]
cf_scores = [rec['cf_score'] for rec in top_5_recs]
cb_scores = [rec['cb_score'] for rec in top_5_recs]
pop_scores = [rec['pop_score'] for rec in top_5_recs]

x = np.arange(len(titles))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - width, cf_scores, width, label='Collaborative Filtering', color='steelblue')
ax.bar(x, cb_scores, width, label='Content-Based', color='coral')
ax.bar(x + width, pop_scores, width, label='Popularity', color='lightgreen')

ax.set_xlabel('Movies', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Hybrid Recommendation Score Breakdown (Top 5)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(titles, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 5.3 Cold-Start User Recommendations

In [ ]:
# Simulate cold-start user (user with very few interactions)
# Find a user with minimal interactions
user_interaction_counts = train_df.groupby('user_id').size()
cold_start_user = user_interaction_counts[user_interaction_counts < 3].index[0]

print(f"\n🆕 Cold-Start User: {cold_start_user}")
print(f"Number of interactions: {user_interaction_counts[cold_start_user]}")

cold_start_recs = hybrid_recommender.recommend(cold_start_user, n_recommendations=10)

print(f"\n{'='*80}")
print(f"🎬 COLD-START RECOMMENDATIONS FOR USER {cold_start_user}")
print(f"{'='*80}\n")

for i, rec in enumerate(cold_start_recs, 1):
    print(f"{i}. {rec['title']}")
    print(f"   Genre: {rec['genre']}")
    print(f"   Hybrid Score: {rec['hybrid_score']:.3f}")
    print(f"   Explanation: {rec['explanation']}")
    print()

### 5.4 Explainable Recommendations

In [ ]:
# Get detailed explanation for a specific recommendation
user_id = 1
movie_id = recommendations[0]['movie_id']

explanation = hybrid_recommender.explain_recommendation(user_id, movie_id)

print(f"\n{'='*80}")
print(f"📖 DETAILED EXPLANATION")
print(f"{'='*80}\n")

print(f"Movie: {explanation['title']}")
print(f"Genre: {explanation['genre']}")
print(f"\nScores:")
for score_type, score_value in explanation['scores'].items():
    print(f"  {score_type}: {score_value:.3f}")

print(f"\nWeights:")
for weight_type, weight_value in explanation['weights'].items():
    print(f"  {weight_type}: {weight_value}")

print(f"\nReasons:")
for reason in explanation['reasons']:
    print(f"  • {reason}")

---

## 6. Evaluation <a id="evaluation"></a>

In [ ]:
# Initialize evaluator
evaluator = RecommenderEvaluator()

# Evaluate hybrid recommender
results = evaluator.evaluate_recommender(hybrid_recommender, test_df, movies_df, k=10)

### 6.1 Visualize Evaluation Metrics

In [ ]:
# Create visualization of metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Ranking metrics
ranking_metrics = ['precision@k', 'recall@k', 'f1@k', 'map@k']
ranking_values = [results[m] for m in ranking_metrics]
ranking_labels = ['Precision@10', 'Recall@10', 'F1@10', 'MAP@10']

axes[0, 0].bar(ranking_labels, ranking_values, color=['steelblue', 'coral', 'lightgreen', 'gold'])
axes[0, 0].set_title('Ranking Metrics', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_ylim(0, 1)
axes[0, 0].grid(axis='y', alpha=0.3)

# Rating prediction metrics
rating_metrics = ['rmse', 'mae']
rating_values = [results[m] for m in rating_metrics]
rating_labels = ['RMSE', 'MAE']

axes[0, 1].bar(rating_labels, rating_values, color=['tomato', 'orange'])
axes[0, 1].set_title('Rating Prediction Metrics', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Error')
axes[0, 1].grid(axis='y', alpha=0.3)

# Catalog metrics
catalog_metrics = ['coverage', 'diversity']
catalog_values = [results[m] for m in catalog_metrics]
catalog_labels = ['Coverage', 'Diversity']

axes[1, 0].bar(catalog_labels, catalog_values, color=['purple', 'teal'])
axes[1, 0].set_title('Catalog Metrics', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].grid(axis='y', alpha=0.3)

# Summary text
axes[1, 1].axis('off')
summary_text = f"""
EVALUATION SUMMARY
{'='*40}

Ranking Performance:
  • Precision@10: {results['precision@k']:.4f}
  • Recall@10: {results['recall@k']:.4f}
  • F1@10: {results['f1@k']:.4f}
  • MAP@10: {results['map@k']:.4f}

Rating Prediction:
  • RMSE: {results['rmse']:.4f}
  • MAE: {results['mae']:.4f}

Catalog Coverage:
  • Coverage: {results['coverage']:.2%}
  • Diversity: {results['diversity']:.4f}

Users Evaluated: {results['n_users_evaluated']}
"""

axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                verticalalignment='center')

plt.tight_layout()
plt.show()

---

## 7. Real-World Improvements <a id="improvements"></a>

### 7.1 Deep Learning Embeddings

**Neural Collaborative Filtering (NCF):**
```python
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from tensorflow.keras.models import Model

# User and item embeddings
user_input = Input(shape=(1,))
item_input = Input(shape=(1,))

user_embedding = Embedding(n_users, 50)(user_input)
item_embedding = Embedding(n_items, 50)(item_input)

user_vec = Flatten()(user_embedding)
item_vec = Flatten()(item_embedding)

# Concatenate and add dense layers
concat = Concatenate()([user_vec, item_vec])
dense1 = Dense(128, activation='relu')(concat)
dense2 = Dense(64, activation='relu')(dense1)
output = Dense(1, activation='sigmoid')(dense2)

model = Model([user_input, item_input], output)
model.compile(optimizer='adam', loss='mse')
```

**BERT for Content Descriptions:**
```python
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Get embeddings for movie descriptions
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()
```

### 7.2 Real-Time Recommendations

**Architecture:**
```
User Action → Kafka → Stream Processor → Update Model → Redis Cache → API
```

**Implementation Sketch:**
```python
# Kafka producer for user events
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Send user interaction event
event = {
    'user_id': 123,
    'movie_id': 456,
    'action': 'watch',
    'timestamp': datetime.now().isoformat()
}
producer.send('user-interactions', event)

# Redis for caching recommendations
import redis

redis_client = redis.Redis(host='localhost', port=6379, db=0)

# Cache recommendations
def cache_recommendations(user_id, recommendations):
    key = f'recs:user:{user_id}'
    redis_client.setex(key, 3600, json.dumps(recommendations))  # 1 hour TTL
```

### 7.3 Explainable AI

**Enhanced Explanations:**
```python
def generate_rich_explanation(user_id, movie_id):
    # Get user's watch history
    user_history = get_user_history(user_id)
    
    # Find similar watched movies
    similar_watched = find_similar_movies(movie_id, user_history)
    
    # Generate explanation
    if similar_watched:
        explanation = f"Because you watched {similar_watched[0]['title']}"
    elif is_trending(movie_id):
        explanation = "Trending now in your region"
    elif matches_genre_preference(user_id, movie_id):
        genre = get_movie_genre(movie_id)
        explanation = f"Based on your interest in {genre}"
    else:
        explanation = "Popular among users like you"
    
    return explanation
```

### 7.4 Context-Aware Recommendations

**Contextual Factors:**
- Time of day (morning: news/documentaries, evening: movies)
- Day of week (weekend: longer content)
- Device (mobile: shorter content, TV: full movies)
- Location (regional preferences)
- Mood (inferred from recent behavior)

```python
def context_aware_recommend(user_id, context):
    base_recs = hybrid_recommender.recommend(user_id, n_recommendations=50)
    
    # Apply contextual filters
    if context['time_of_day'] == 'morning':
        # Boost short content
        base_recs = boost_by_duration(base_recs, max_duration=30)
    
    if context['device'] == 'mobile':
        # Prefer episodic content
        base_recs = boost_by_type(base_recs, content_type='series')
    
    if context['weekend']:
        # Boost movies and binge-worthy series
        base_recs = boost_by_type(base_recs, content_type='movie')
    
    return base_recs[:10]
```

### 7.5 A/B Testing Framework

**Multi-Armed Bandit:**
```python
class EpsilonGreedy:
    def __init__(self, epsilon=0.1):
        self.epsilon = epsilon
        self.model_rewards = {}  # model_name -> [rewards]
    
    def select_model(self, available_models):
        if np.random.random() < self.epsilon:
            # Explore: random model
            return np.random.choice(available_models)
        else:
            # Exploit: best performing model
            avg_rewards = {
                model: np.mean(self.model_rewards.get(model, [0]))
                for model in available_models
            }
            return max(avg_rewards, key=avg_rewards.get)
    
    def update(self, model_name, reward):
        if model_name not in self.model_rewards:
            self.model_rewards[model_name] = []
        self.model_rewards[model_name].append(reward)
```

---

## 📝 Summary

### What We Built:
✅ **Hybrid Recommender System** combining CF, CB, and Popularity

✅ **Cold-Start Handling** for new users and items

✅ **Comprehensive Evaluation** with multiple metrics

✅ **Explainable Recommendations** with reasoning

✅ **Production-Ready Architecture** with scalability considerations

### Key Takeaways:
1. **Hybrid approaches** outperform individual methods
2. **Cold-start** requires special handling strategies
3. **Evaluation** needs multiple metrics (accuracy, diversity, coverage)
4. **Explainability** improves user trust and engagement
5. **Real-time** and **context-aware** features enhance user experience

### Next Steps:
- Deploy as REST API using Flask/FastAPI
- Implement real-time updates with Kafka
- Add deep learning embeddings
- Build A/B testing framework
- Scale with Apache Spark for big data

---

**🎓 Perfect for:**
- College projects and assignments
- Hackathons and competitions
- Startup MVP development
- Learning recommendation systems
- Portfolio projects

**📧 Questions?** Open an issue on GitHub!

---

*Built with ❤️ for the next generation of OTT platforms*